No WS4 command here uses shell magic: every one goes through this helper, which is also
how you would drive the command line from any script.

This one is a **command-line** session: the operators are tuned with flags.


In [1]:
import subprocess


def run(command: str) -> None:
    """Run a WS4 command, printing what it printed."""
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    print((result.stdout or result.stderr).strip())
    result.check_returncode()

# Patch size-distribution targets

A patch is a connected set of adjacent blocks that share a criterion. On the sample estate the
criterion is a fresh opening, so the patch accounts say how the harvesting is laid out on the
map rather than how much of it there is. This notebook sets a **ramp** — every period must put
more of its opening area into the 50-150 ha class — and watches the search follow it.

## Give the patch operators a weight

`merge` treats the negative space between two proximal patches so a single treatment joins them,
`split` clears an articulation point, and `cluster` rechooses a connected set of blocks. They ship
disabled, so the search has no shape-aware moves until they are weighted.

In [2]:
from pathlib import Path

import pandas as pd

work = Path("tmp/examples/patch-targets")
work.mkdir(parents=True, exist_ok=True)
(work / "targets.yaml").write_text('targets:\n  - account: product.Harvested.CC\n    year: 0\n    minimum: 20000\n    period_width: 10\n  - account: patch.opening.50_150.size\n    year: 0\n    minimum: 25\n    min_weight: 2000\n    shape: non_linear\n    period_width: 10\n  - account: patch.opening.50_150.size\n    year: 10\n    minimum: 50\n    min_weight: 2000\n    shape: non_linear\n    period_width: 10\n  - account: patch.opening.50_150.size\n    year: 20\n    minimum: 75\n    min_weight: 2000\n    shape: non_linear\n    period_width: 10\n')

490

In [3]:
run("ws4 optimize --dataset tmp/examples/estate/compiled"
    " --targets tmp/examples/patch-targets/targets.yaml --years 100 --iterations 20000"
    " --topology examples/estate/topology.csv --patch-accounts examples/estate/patches.yaml"
    " --merge-weight 0.5 --split-weight 0.2 --cluster-weight 0.3 --threads 1"
    " --out tmp/examples/patch-targets/run")

                              Optimization Summary                              
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ key                 ┃ value                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ incumbent_objective │ -2.7284841053187847e-10                                │
│ iterations          │ 20000                                                  │
│ patience            │ 0                                                      │
│ seed                │ 42                                                     │
│ metaheuristic       │ sa                                                     │
│ starts              │ 1                                                      │
│ blocks_scheduled    │ 35                                                     │
│ operators           │ ['toggle', 'shift', 'swap', 'lns', 'cluster', 'merge', │
│                     │ 'spl

## Read the run

In [4]:
print(pd.read_csv(work / "run" / "report_targets.csv").to_string(index=False))

                  account  year        value  minimum  maximum       penalty status
     product.Harvested.CC     0 20360.914286    20000      NaN      0.000000    met
patch.opening.50_150.size     0    54.039965       25      NaN      0.000000    met
patch.opening.50_150.size    10    54.039965       50      NaN      0.000000    met
patch.opening.50_150.size    20    54.039965       75      NaN 878646.113643  below


## Watch the classes per period

`patch_accounts.csv` has one column per size class and sub-account, one row per period. The
share in the 50-150 ha class is `...50_150.size`, in percent.

In [5]:
accounts = pd.read_csv(work / "run" / "patch_accounts.csv")
shares = accounts[["year", "patch.opening.0_10.size", "patch.opening.10_50.size", "patch.opening.50_150.size"]]
print(shares.to_string(index=False))

 year  patch.opening.0_10.size  patch.opening.10_50.size  patch.opening.50_150.size
  0.0                 0.000000                 45.960035                  54.039965
  1.0                 0.000000                 45.960035                  54.039965
  2.0                 0.000000                 45.960035                  54.039965
  3.0                 0.000000                 45.960035                  54.039965
  4.0                 0.000000                 45.960035                  54.039965
  5.0                 0.000000                 45.960035                  54.039965
  6.0                 0.000000                 45.960035                  54.039965
  7.0                 0.000000                 45.960035                  54.039965
  8.0                 0.000000                 45.960035                  54.039965
  9.0                 0.000000                 45.960035                  54.039965
 10.0                 0.000000                 45.960035                  54

The first three periods are the ramp — 25, then 50, then 75 percent of the opening area in
the 50-150 ha class. The search meets the first two (63.8 percent) and falls short of the third,
which is the honest outcome of a weight of 2000 against a squared miss: 11.2 points of shortfall
cost 249 340. Note what the shares do *not* do: they stay steady across the horizon, because the
40-year rotation keeps refreshing the openings into the same class, which is the structure the
target is there to hold. The run's report prices exactly the shares the search found; compare it
with `patch_metrics.csv` from a simulation of the same schedule, which is the check worth making
on any patch result.

## Where to look next

* `docs/howto/patch-targets.rst` explains the size classes, `search_distance` and the operators.
* `docs/reference/spatial.rst` specifies the topology, the metrics and the opening rules.